# LiveKit Voice Agent Architecture

Rebuild the audio plumbing between a SIP participant and an `AgentSession`: rate conversion, framing, and call routing. Predict each result before running the cell.

In [ ]:
import sys
sys.path.insert(0, '../code')
import numpy as np
import main

for name, rate, ms in main.CALL_PATH:
    print(f"{name:<20} {rate:>6} Hz  {ms} ms -> {main.frame_samples(rate, ms)} samples")

## Resampling is filtering

Downsample a 1 kHz + 6 kHz mix from 48 kHz to 8 kHz. Where does the 6 kHz tone go with and without the anti-alias filter?

In [ ]:
x = main.tone(1_000, 48_000) + main.tone(6_000, 48_000)
for name, y in (('windowed sinc', main.resample(x, 48_000, 8_000)), ('linear interp', main.naive_resample(x, 48_000, 8_000))):
    print(f"{name:<14} 1 kHz {main.tone_level_db(y, 8_000, 1_000):6.1f} dB   2 kHz alias {main.tone_level_db(y, 8_000, 2_000):7.1f} dB")

Try `zeros=4` instead of the default 16. What happens to the alias level and to the number of taps?

In [ ]:
for zeros in (4, 8, 16):
    up, down, h = main.design_resampler(48_000, 8_000, zeros=zeros)
    y = main.resample(main.tone(4_400, 48_000), 48_000, 8_000, zeros=zeros)
    print(f"zeros={zeros:<3} taps={len(h):<4} 4.4 kHz -> 3.6 kHz alias at {main.tone_level_db(y, 8_000, 3_600):6.1f} dB")

## Frames from uneven chunks

In [ ]:
stream = main.FrameStream(sample_rate=16_000, frame_ms=10)
rng = np.random.default_rng(0)
frames = []
for size in rng.integers(50, 900, size=10):
    frames += stream.push(rng.standard_normal(size))
frames += stream.flush()
len(frames), {len(f) for f in frames}

## Routing a call

Add a rule, change the PIN, or hide the phone number, and watch the room and attributes change.

In [ ]:
rules = [
    main.DispatchRule(kind='direct', room_name='vip-line', pin='2468'),
    main.DispatchRule(kind='callee', room_prefix='line-', randomize=False, trunk_ids=('ST_sales',)),
    main.DispatchRule(kind='individual', room_prefix='call-', agent_name='inbound-agent'),
]
for trunk, pin in (('ST_hotline', ''), ('ST_hotline', '2468'), ('ST_sales', '')):
    routed = main.route_call(rules, trunk, '+84901234567', '+842873001234', 'call-1', pin=pin)
    print(trunk, repr(pin), '->', routed['room'], routed['agent_name'] or '(no agent)')